In [1]:
!git clone https://github.com/vimalathithan17/quantum-classification-train.git
%cd /kaggle/working/quantum-classification-train

Cloning into 'quantum-classification-train'...
remote: Enumerating objects: 1224, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 1224 (delta 93), reused 91 (delta 71), pack-reused 1094 (from 1)
Receiving objects: 100% (1224/1224), 1.05 MiB | 4.01 MiB/s, done.
Resolving deltas: 100% (850/850), done.
[Errno 2] No such file or directory: '/kaggle/working/quantum-classification-train'
/workspaces/quantum-classification-train/kaggle_notebooks


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
import wandb

# Set environment variables
os.environ["WANDB_API_KEY"] = "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" # REPLACE THIS

# In this notebook, your input is the Kaggle Dataset you created from 01_Global_Data_Split
os.environ["GLOBAL_TRAIN_DIR"] = "/kaggle/input/your-global-split-dataset/data/global_train" # REPLACE THIS
os.environ["GLOBAL_TEST_DIR"] = "/kaggle/input/your-global-split-dataset/data/global_test" # REPLACE THIS
os.environ["ENCODER_DIR"] = "/kaggle/input/your-global-split-dataset/master_label_encoder" # REPLACE THIS

os.environ["OUTPUT_DIR"] = "/kaggle/working/output"

# Ensure directories exist
!mkdir -p /kaggle/working/output

# Login to wandb
wandb.login(key=os.environ["WANDB_API_KEY"])

In [ ]:
# Copy previous run data if necessary (e.g., Optuna DB, pre-trained models)
# !cp -r /kaggle/input/previous-run-data/* /kaggle/working/output/
print("Setup complete. Ready to run.")

In [ ]:
# 1. Pretrain the contrastive encoder on global_train
!python examples/pretrain_contrastive.py \
    --data_dir $GLOBAL_TRAIN_DIR \
    --output_dir /kaggle/working/output/encoder_models \
    --batch_size 64 --epochs 100 --device cuda

# 2. Extract features for BOTH train and test splits using the trained encoder
!python examples/extract_pretrained_features.py \
    --data_dir $GLOBAL_TRAIN_DIR \
    --global_test_dir $GLOBAL_TEST_DIR \
    --model_path /kaggle/working/output/encoder_models/best_contrastive_model.pt \
    --output_dir /kaggle/working/output/pretrained_features \
    --device cuda

# 3. Zip the extracted features so they can be turned into a Kaggle Dataset for notebooks 03/04
!zip -qr /kaggle/working/pretrained_embeddings.zip /kaggle/working/output/pretrained_features